In [0]:
from pyspark.sql import functions as F
T = lambda n: spark.table(f"workspace.bronze.{n}")
stores, customers, orders, items, products = (T(x) for x in
    ["stores", "customers", "orders", "order_items", "products"])

def show(title, df):
    print(f"\n--- {title}: {df.count()} rows ---")
    df.show(5, truncate=40)

# 1. Duplicate primary keys
show("dup store_id",       stores.groupBy("store_id").count().filter("count>1"))
show("dup customer_id",    customers.groupBy("customer_id").count().filter("count>1"))
show("dup order_id",       orders.groupBy("order_id").count().filter("count>1"))
show("dup order_item_id",  items.groupBy("order_item_id").count().filter("count>1"))

# 2. Orphan foreign keys
show("orders -> unknown customer", orders.join(customers, "customer_id", "left_anti"))
show("orders -> unknown store",    orders.join(stores, "store_id", "left_anti"))
show("items -> unknown order",     items.join(orders, "order_id", "left_anti"))
show("items -> unknown product",   items.join(products.withColumnRenamed("id","product_id"), "product_id", "left_anti"))

# 3. Invalid values
show("quantity <= 0",     items.filter("quantity <= 0"))
show("unit_price <= 0",   items.filter("unit_price <= 0"))
show("orders in future",  orders.filter(F.col("order_date") > F.current_timestamp()))

# 4. Category values and formats
orders.groupBy("order_status").count().show()
orders.groupBy("payment_mode").count().show()
customers.filter(~F.col("email").rlike(r"^[^@\s]+@[^@\s]+\.[^@\s]+$")).show(5, truncate=40)

# 5. Orders with no items
show("orders without items", orders.join(items, "order_id", "left_anti"))


--- dup store_id: 0 rows ---
+--------+-----+
|store_id|count|
+--------+-----+
+--------+-----+


--- dup customer_id: 0 rows ---
+-----------+-----+
|customer_id|count|
+-----------+-----+
+-----------+-----+


--- dup order_id: 0 rows ---
+--------+-----+
|order_id|count|
+--------+-----+
+--------+-----+


--- dup order_item_id: 0 rows ---
+-------------+-----+
|order_item_id|count|
+-------------+-----+
+-------------+-----+


--- orders -> unknown customer: 0 rows ---
+-----------+--------+--------+----------+------------+------------+----------+------------+------------+
|customer_id|order_id|store_id|order_date|payment_mode|order_status|updated_at|_source_file|_ingested_at|
+-----------+--------+--------+----------+------------+------------+----------+------------+------------+
+-----------+--------+--------+----------+------------+------------+----------+------------+------------+


--- orders -> unknown store: 0 rows ---
+--------+--------+-----------+----------+------------